# 13. Efficient Training: AMP, Memory, Data Loading, Profiling, and Compilation

**Author:** Md. Mobarak Karim, Ph.D.  
**Level:** Beginner → research-practical  
**Course:** Deep Learning for Optical Imaging

This notebook teaches a measurement-first optimization workflow rather than random speed tricks.

> **How to study this notebook:** read the explanation first, predict what the code should do, run it, change one parameter, and explain why the result changed.


## Learning objectives

- Profile the pipeline conceptually
- Use AMP appropriately
- Reason about batch size and accumulation
- Tune data loading
- Choose memory strategies
- Understand profiler and torch.compile
- Document speed/quality trade-offs


## Mind map

```mermaid
mindmap
  root((Optimization))
    Measure
      Step time
      GPU utilization
      Memory
    Data
      Workers
      Pin memory
      Storage
    Precision
      AMP
      Grad scaling
    Memory
      Batch
      Accumulation
      Checkpointing
    Compute
      Profiler
      torch.compile
      Distributed
    Validate
      Metric unchanged
      Numerical stability

```


## 1. Optimization starts after correctness

Do not make a broken pipeline faster.

Order:

```text
correctness
→ valid split/metric
→ tiny-set overfit
→ reproducible baseline
→ measure time/memory
→ remove bottleneck
→ mixed precision
→ batch/memory strategy
→ profiler
→ compile if beneficial
→ distributed training only if justified
```


## 2. Measure before guessing

Separate:
- data loading time;
- host-to-device transfer;
- forward pass;
- backward pass;
- optimizer step;
- validation/inference;
- augmentation.

GPU utilization can be low because CPU/data loading is slow, not because the model needs a different architecture.


## 3. Automatic mixed precision (AMP)

Modern PyTorch supports autocasting and gradient scaling.

Concept:
- safe operations run in lower precision;
- sensitive operations remain higher precision;
- gradient scaling helps avoid underflow in fp16 training.

Use AMP when supported, then verify numerical behavior and validation quality.


In [ ]:
import torch

# Demonstration pattern; meaningful speedup requires a CUDA GPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# In current PyTorch, autocast is device-type aware:
with torch.autocast(device_type=device.type, enabled=(device.type=="cuda")):
    x = torch.randn(4, 4, device=device)
    y = x @ x
print(y.dtype)


## 4. Batch size and gradient accumulation

Larger batch can improve throughput but uses more memory.

Gradient accumulation approximates a larger effective batch:

```text
effective_batch = physical_batch × accumulation_steps
```

When using it, divide loss appropriately and call optimizer step only after the accumulation interval.


## 5. DataLoader tuning

Potential controls:
- `num_workers`;
- `pin_memory` for CUDA workflows;
- persistent workers;
- prefetching;
- caching only when safe/feasible;
- avoiding very slow Python transforms.

Benchmark on your own storage/CPU/GPU. More workers are not always faster.


## 6. Memory strategies

Before buying/scaling hardware:
- reduce batch size;
- use mixed precision;
- crop/patch;
- reduce input resolution if scientifically acceptable;
- gradient accumulation;
- activation/gradient checkpointing;
- smaller model;
- 2-D/2.5-D baseline instead of 3-D.

Every memory-saving choice can affect scientific performance, so validate it.


## 7. Profiling

`torch.profiler` can show CPU/GPU operator timing, memory, and shapes.

Use profiling to identify:
- expensive kernels;
- excessive data copies;
- unexpected CPU operations;
- shape-dependent bottlenecks.

Profile a short representative workload, not an entire 100-epoch run.


## 8. torch.compile

`torch.compile(model)` may improve performance by compiling graphs, but:
- first iterations can be slower due to compilation;
- graph breaks can reduce benefit;
- dynamic shapes may matter;
- speedup is workload/hardware dependent.

Benchmark warm and steady-state execution. Keep an uncompiled reference path for debugging.


## 9. Optimization report template

```text
Model:
Input shape:
Batch:
GPU:
Precision:
DataLoader workers:
Baseline step time:
Peak memory:
Change:
New step time:
New peak memory:
Validation metric before:
Validation metric after:
Conclusion:
```

An optimization is useful only if it improves the resource target without invalidating scientific performance.


## End-of-notebook checklist

Before moving on, you should be able to explain the main ideas **without looking at the code**. If you cannot explain why a method, loss, split, or metric is appropriate, repeat the relevant section before using it in research.
